# E337 - Grupo 1 - Trabajo Práctico 2
Integrantes: Constanza María Efkhanian - Julián Fabrizio Notario - María Florencia Pascale

## Bibliotecas

In [37]:
import os
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib as mpl

## Directorio

In [38]:
directorio = "/Users/mariapascale/Documents/E337 | Big Data/E337_Grupo1/TP2"
os.chdir(directorio)
print(os.getcwd())

/Users/mariapascale/Documents/E337 | Big Data/E337_Grupo1/TP2


## Limpieza de la base de datos

Nuestro objetivo aquí es limpiar y concatenar los dos data frames con todas sus variables. Para este paso, reutilizamos gran parte del código que habíamos escrito en el TP1, pero también hicimos algunos cambios; el código anterior estaba especificado para que solo hubiese 15 variables en el data frame nuevo, y aquí queremos incluir todas. 

In [39]:
# Abrimos las bases de datos
df_2005_raw = pd.read_stata('usu_individual_3105.dta', convert_categoricals=False, convert_missing=False)
df_2005_raw['EMPLEO'] = np.nan # Creamos la variable EMPLEO en la base del 2005 porque no está

df_2025_raw = pd.read_excel('usu_individual_T325.xls')

# Pasamos todas las variables a mayúsculas para armonizarlas:
df_2005_raw.columns = df_2005_raw.columns.str.upper().str.strip()
df_2025_raw.columns = df_2025_raw.columns.str.upper().str.strip()

# Filtramos por GBA
df_2005 = df_2005_raw[df_2005_raw['REGION'] == 1].copy()
df_2025 = df_2025_raw[df_2025_raw['REGION'] == 1].copy()

print(f"\nFiltro GBA")
print(f"  2005: {len(df_2005):,} observaciones")
print(f"  2025: {len(df_2025):,} observaciones")

# Ahora, forzamos el año:
df_2005['ANO4'] = 2005
df_2025['ANO4'] = 2025



Filtro GBA
  2005: 9,420 observaciones
  2025: 7,447 observaciones


Luego, toca convertir las variables categóricas a un mismo formato. Esta parte la hicimos con ayuda de la IA. 

In [40]:
# ============================================================
# ARMONIZACIÓN DE FORMATOS
# ============================================================

# --- PROBLEMA 1: Variables de ingresos con -9 sin limpiar en 2025 ---
vars_ingreso_extra = [
    'PP08D1', 'PP08F1', 'PP08F2', 'PP08J1', 'PP08J2', 'PP08J3',
    'TOT_P12', 'P47T', 'T_VI',
    'V3_M', 'V4_M', 'V8_M', 'V9_M', 'V10_M', 'V12_M', 'V18_M'
]

for col in vars_ingreso_extra:
    if col in df_2025.columns:
        df_2025.loc[df_2025[col] == -9, col] = np.nan

print("[1] -9 en variables de ingreso → NaN (2025)")


# --- PROBLEMA 2: Variables de código string vs float ---
def normalizar_codigo(serie):
    """Convierte string o float a string entero limpio. NaN se preserva."""
    def _conv(x):
        if pd.isna(x):
            return np.nan
        try:
            val = str(x).strip()
            if val == '' or val in ('nan', 'None'):
                return np.nan
            return str(int(float(val)))
        except:
            return np.nan
    return serie.apply(_conv)

vars_codigo = [
    'CH14', 'CH15_COD', 'CH16_COD',
    'PP04B_COD', 'PP04D_COD', 'PP11B_COD', 'PP11D_COD'
]

for col in vars_codigo:
    if col in df_2005.columns:
        df_2005[col] = normalizar_codigo(df_2005[col])
    if col in df_2025.columns:
        df_2025[col] = normalizar_codigo(df_2025[col])

print("[2] Variables de código normalizadas a string entero limpio")


# --- PROBLEMA 3: Variables de deciles con padding inconsistente ---
vars_deciles = [
    'DECOCUR', 'IDECOCUR', 'RDECOCUR', 'GDECOCUR', 'PDECOCUR', 'ADECOCUR',
    'DECINDR', 'IDECINDR', 'RDECINDR', 'GDECINDR', 'PDECINDR', 'ADECINDR',
    'DECIFR',  'IDECIFR',  'RDECIFR',  'GDECIFR',  'PDECIFR',  'ADECIFR',
    'DECCFR',  'IDECCFR',  'RDECCFR',  'GDECCFR',  'PDECCFR',  'ADECCFR'
]

for col in vars_deciles:
    if col in df_2005.columns:
        df_2005[col] = normalizar_codigo(df_2005[col])
    if col in df_2025.columns:
        df_2025[col] = normalizar_codigo(df_2025[col])

print("[3] Variables de deciles normalizadas")


# --- VERIFICACIÓN ---
print("\nVerificación post-armonización:")
for col in ['CH14', 'DECOCUR', 'PP04B_COD']:
    v05 = set(df_2005[col].dropna().unique()) if col in df_2005.columns else set()
    v25 = set(df_2025[col].dropna().unique()) if col in df_2025.columns else set()
    comunes = v05 & v25
    print(f"  {col}: {len(comunes)} valores en común entre bases")

[1] -9 en variables de ingreso → NaN (2025)
[2] Variables de código normalizadas a string entero limpio
[3] Variables de deciles normalizadas

Verificación post-armonización:
  CH14: 12 valores en común entre bases
  DECOCUR: 11 valores en común entre bases
  PP04B_COD: 29 valores en común entre bases


Ahora toca eliminar los valores sin sentido de la base de datos. Para esto, utilizamos ayuda de Claude. Le enviamos los diccionarios de ambos años y le pedimos que identificara potenciales valores sin sentido. A continuación, le pedimos un código que reemplazara los valores sin sentido por valores NaN. El chat completo de esta sección y demás chats en los que hayamos interactuado con la IA pueden hallarse en la parte III del informe. 

In [41]:
# --- CH06: edad inválida ---
df_2005['CH06'] = df_2005['CH06'].replace(-1, np.nan)
df_2025['CH06'] = df_2025['CH06'].replace(-1, np.nan)

# --- CH07: estado civil Ns/Nr ---
df_2005['CH07'] = df_2005['CH07'].replace(9, np.nan)
df_2025['CH07'] = df_2025['CH07'].replace(9, np.nan)

# --- CH08: cobertura médica Ns/Nr (9 = Ns/Nr; 123 es válido) ---
df_2005['CH08'] = df_2005['CH08'].replace(9, np.nan)
df_2025['CH08'] = df_2025['CH08'].replace(9, np.nan)

# --- NIVEL_ED: nivel educativo Ns/Nr ---
df_2005['NIVEL_ED'] = df_2005['NIVEL_ED'].replace(9, np.nan)
df_2025['NIVEL_ED'] = df_2025['NIVEL_ED'].replace(9, np.nan)

# --- CAT_OCUP: categoría ocupacional Ns/Nr ---
df_2005['CAT_OCUP'] = df_2005['CAT_OCUP'].replace(9, np.nan)
df_2025['CAT_OCUP'] = df_2025['CAT_OCUP'].replace(9, np.nan)

# --- PP04A: tipo de empresa (0 = no aplica, 9 = Ns/Nr) ---
df_2005['PP04A'] = df_2005['PP04A'].replace({0: np.nan, 9: np.nan})
df_2025['PP04A'] = df_2025['PP04A'].replace({0: np.nan, 9: np.nan})

# --- P21: ingreso ocupación principal ---
# 2005: no había -9 según diccionario, pero por las dudas limpiamos negativos
# 2025: -9 = no respuesta (confirmado en diccionario)
df_2005.loc[df_2005['P21'] < 0, 'P21'] = np.nan
df_2025.loc[df_2025['P21'] < 0, 'P21'] = np.nan

# --- IPCF: ingreso per cápita familiar ---
# Mismo criterio que P21
df_2005.loc[df_2005['IPCF'] < 0, 'IPCF'] = np.nan
df_2025.loc[df_2025['IPCF'] < 0, 'IPCF'] = np.nan

# --- PP06C y PP06D: ingresos de independientes ---
# -7 = no tenía esa ocupación, -8 = sin ingresos ese mes, -9 = no respuesta
for col in ['PP06C', 'PP06D']:
    for df in [df_2005, df_2025]:
        if col in df.columns:
            df[col] = df[col].replace({-7: np.nan, -8: np.nan, -9: np.nan})

# --- EMPLEO: formalidad laboral (solo existe en 2025; en 2005 ya es todo NaN) ---
if 'EMPLEO' in df_2025.columns:
    df_2025['EMPLEO'] = df_2025['EMPLEO'].replace(9, np.nan)

# --- Regla general: 9/99/999/9999 = Ns/Nr en variables categóricas de 1 dígito ---
# Las variables de ingresos (montos) son la excepción — ya tratadas arriba.
# Aplicar con criterio: solo donde el máximo valor válido lo permite.

In [42]:
# Concatenamos los dataframes
df_final = pd.concat([df_2005, df_2025], ignore_index=True)

# Finalmente, guardamos la base df_final combinada como csv. 

df_final.to_csv('eph_gba_merged_full.csv', index=False)
print("\nBase guardada como 'eph_gba_merged_full.csv'")



Base guardada como 'eph_gba_merged_full.csv'


### Ejercicio 1


### Ejercicio 2

### Ejercicio 3

### Ejercicio 4

### Ejercicio 5

# Parte II: Métodos No Supervisados

### Ejercicio 1

## PCA

### Ejercicio 2

### Ejercicio 3

### Ejercicio 4

## Cluster

### Ejercicio 5

#### Inciso a

#### Inciso b

### Ejercicio 6

### Ejercicio 7

# Parte III: Herramientas de IA y reflexión final


Realizado en documento de LaTeX. 